In [1]:
# Core packages
import os  # for file and path operations
import warnings  # to suppress warnings
warnings.filterwarnings("ignore")

# Data manipulation
import pandas as pd  # for handling dataframes
import numpy as np  # for numerical operations

# Visualization
import matplotlib.pyplot as plt  # for general plotting
import seaborn as sns  # for enhanced statistical plotting
from matplotlib.pylab import rcParams  # for customizing plot size and appearance

# Machine learning - preprocessing & model evaluation
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score  # data splitting and cross-validation
from sklearn.preprocessing import StandardScaler  # feature standardization

# Machine learning - models
from sklearn.tree import DecisionTreeClassifier, plot_tree  # decision tree model and plotting
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier   # random forest model

# Machine learning - metrics
from sklearn.metrics import (
    ConfusionMatrixDisplay,  # plot confusion matrix
    classification_report,  # detailed classification report
    confusion_matrix,  # confusion matrix array
    accuracy_score,  # accuracy metric
    precision_score,  # precision metric
    recall_score,  # recall metric
    f1_score  # F1 score
)

# PyTorch - deep learning
import torch  # base PyTorch package
import torch.nn as nn  # neural network layers
import torch.optim as optim  # optimizers (e.g., Adam, SGD)

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

In [10]:
# Load data
full = pd.read_csv("C:/Users/annaw/Desktop/DataScience/Datasets/full_dataset.csv", dtype={91: str})
print(f"X_train shape: {full.shape}")

# Drop columns with >20% missing
full_clean = full.dropna(thresh=len(full)*0.8, axis=1)  
print(full_clean['avalancheDay1'].value_counts())

# Save year column before dropping others
years = full_clean['year']
print(years)
# Drop rows with any missing values
full_clean = full_clean.dropna()

# Select only numeric columns
full_clean = full_clean.select_dtypes(include=[np.number])

# Drop unwanted columns (keep year for now)
if 'datum' in full_clean.columns:
    full_clean = full_clean.drop(columns=['datum'])

# Add year back to use for splitting
full_clean['year'] = years.loc[full_clean.index]

# Define features and target
X = full_clean.drop(columns=["avalancheDay1"])
y = full_clean["avalancheDay1"]

# Split using year == 22 for val/test, rest for training
X_temp = X[X['year'] != 2022].drop(columns=['year'])
y_temp = y[X['year'] != 2022]

X_test_meta = X[X['year'] == 2022].drop(columns=['year'])
y_test_meta = y[X['year'] == 2022]

# Optional: split year 22 set further into val and test
X_train_base, X_train_meta, y_train_base, y_train_meta = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

#I need to split the data carefully and save some testing data for the stacking model 

# 1) Use one dataset to train the BASE models (X_train_base, y_train_base)
# 2) Use another dataset to train the meta-model on base model predictions (X_train_meta, y_train_meta)
# 3) Usa a third, untouched dataset to test the full stacked model (X_test_meta, y_test_meta)

# Standardize
scaler = StandardScaler()
X_train_base_scaled = scaler.fit_transform(X_train_base)
X_train_meta_scaled = scaler.transform(X_train_meta)
X_test_meta_scaled = scaler.transform(X_test_meta)

X_train shape: (11362, 95)
avalancheDay1
0    10638
1      724
Name: count, dtype: int64
0        2020
1        2020
2        2020
3        2020
4        2020
         ... 
11357    2022
11358    2021
11359    2022
11360    2022
11361    2022
Name: year, Length: 11362, dtype: int64


In [11]:
print("X_train_base_scaled:", X_train_base_scaled.shape)
print("X_train_meta_scaled:", X_train_meta_scaled.shape)
print("X_test_meta_scaled:", X_test_meta_scaled.shape)

X_train_base_scaled: (6310, 84)
X_train_meta_scaled: (2104, 84)
X_test_meta_scaled: (1879, 84)


In [12]:
# Define NN model
class AvalancheNet(nn.Module):
    def __init__(self, input_dim):
        super(AvalancheNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64) # first hidden layer
        self.dropout1 = nn.Dropout(0.3) # regularization
        self.fc2 = nn.Linear(64, 32) # second hidden layer
        self.dropout2 = nn.Dropout(0.3) 
        self.fc3 = nn.Linear(32, 1) # output layer (1 logit for binary classification)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

In [13]:
#Wrap the pytorch model for stacking
class TorchNNWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, input_dim, threshold=0.5, epochs=100, lr=0.001, pos_weight=5.0):
        self.threshold = threshold
        self.epochs = epochs
        self.lr = lr
        self.pos_weight = pos_weight  
        self.input_dim = input_dim

    def _build_model(self):
        return AvalancheNet(self.input_dim)

    def fit(self, X, y):
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y.to_numpy().astype(np.float32)).unsqueeze(1)

        self.model = self._build_model()
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([self.pos_weight]))

        for _ in range(self.epochs):
            self.model.train()
            optimizer.zero_grad()
            outputs = self.model(X_tensor)
            loss = criterion(outputs, y_tensor)
            loss.backward()
            optimizer.step()
        return self

    def predict_proba(self, X):
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.tensor(X, dtype=torch.float32)
            logits = self.model(X_tensor)
            probs = torch.sigmoid(logits).numpy().flatten()
        return np.vstack([1 - probs, probs]).T

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > self.threshold).astype(int)

In [14]:
#F1 optimized RF
rf_custom_F1 = RandomForestClassifier(
    class_weight=None,
    max_depth=20,
    max_features=None,
    min_samples_leaf=5,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

#Recall optimized RF

rf_custom_Recall = RandomForestClassifier(
    class_weight='balanced',
    max_depth=10,
    max_features="sqrt",
    min_samples_leaf=200,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [15]:
# 1. Train base models on base set
rf_custom_F1.fit(X_train_base_scaled, y_train_base)
rf_custom_Recall.fit(X_train_base_scaled, y_train_base)
nn_wrapper = TorchNNWrapper(input_dim=X_train_base_scaled.shape[1], threshold=0.5, epochs=200)
nn_wrapper.fit(X_train_base_scaled, y_train_base)

# 2. Predict meta-set using pretrained base models
meta_features = np.column_stack([
    rf_custom_F1.predict_proba(X_train_meta_scaled)[:, 1],
    rf_custom_Recall.predict_proba(X_train_meta_scaled)[:, 1],
    nn_wrapper.predict_proba(X_train_meta_scaled)[:, 1]
])

# Train meta-learner
meta_learner = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3)
meta_learner.fit(meta_features, y_train_meta)

# 4. Evaluate on final test set
test_features = np.column_stack([
    rf_custom_F1.predict_proba(X_test_meta_scaled)[:, 1],
    rf_custom_Recall.predict_proba(X_test_meta_scaled)[:, 1],
    nn_wrapper.predict_proba(X_test_meta_scaled)[:, 1]
])

y_pred_stack = meta_learner.predict(test_features)

# Evaluate on meta test set
print("\n=== Stacked Model Performance ===")
print(classification_report(y_test_meta, y_pred_stack))
print(confusion_matrix(y_test_meta, y_pred_stack))


=== Stacked Model Performance ===
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1819
           1       0.65      0.88      0.75        60

    accuracy                           0.98      1879
   macro avg       0.83      0.93      0.87      1879
weighted avg       0.99      0.98      0.98      1879

[[1791   28]
 [   7   53]]


In [1]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

# 1. Train base models on base set
rf_custom_F1.fit(X_train_base_scaled, y_train_base)
rf_custom_Recall.fit(X_train_base_scaled, y_train_base)
nn_wrapper = TorchNNWrapper(input_dim=X_train_base_scaled.shape[1], threshold=0.5, epochs=200)
nn_wrapper.fit(X_train_base_scaled, y_train_base)

# 2. Predict meta-set using pretrained base models
meta_features = np.column_stack([
    rf_custom_F1.predict_proba(X_train_meta_scaled)[:, 1],
    rf_custom_Recall.predict_proba(X_train_meta_scaled)[:, 1],
    nn_wrapper.predict_proba(X_train_meta_scaled)[:, 1]
])

meta_scaler = StandardScaler()
meta_features_scaled = meta_scaler.fit_transform(meta_features)

# Train meta-learner
meta_learner = LogisticRegressionCV(cv=5, max_iter=1000, scoring='f1')
meta_learner.fit(meta_features, y_train_meta)

# 4. Evaluate on final test set
test_features = np.column_stack([
    rf_custom_F1.predict_proba(X_test_meta_scaled)[:, 1],
    rf_custom_Recall.predict_proba(X_test_meta_scaled)[:, 1],
    nn_wrapper.predict_proba(X_test_meta_scaled)[:, 1]
])

y_pred_stack = meta_learner.predict(test_features)

# Evaluate on meta test set
print("\n=== Logistic Regression Meta Model ===")
print(classification_report(y_test_meta, y_pred_stack))

# Evaluate
cm_staked = confusion_matrix(y_test_meta, y_pred_stack)

print(cm_staked)

NameError: name 'rf_custom_F1' is not defined

In [ ]:
#Why might the temporal split perform little worse?

#Temporal splits are harder.
#They simulate real-world generalization, where future data may not follow the same distribution as past data.

#Random splits often leak future signals and give optimistically biased results
# especially in time-dependent systems like weather or avalanche risk